# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Acacia21-code/FlyRankAI-ML-Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Acacia21-code/FlyRankAI-ML-Week1.git
%cd FlyRankAI-ML-Week1/work/notebooks

fatal: destination path 'FlyRankAI-ML-Week1' already exists and is not an empty directory.
/content/FlyRankAI-ML-Week1/work/notebooks


In [2]:
import os
print("Current working directory:", os.getcwd())
print(os.listdir("."))

Current working directory: /content/FlyRankAI-ML-Week1/work/notebooks
['w06_validation_audit.ipynb', 'w04_baseline_score.ipynb', 'w07_action_playbook.ipynb', 'capstone.ipynb', 'w03_data_contract.ipynb', 'w04_signal_audit.ipynb', 'w01_research_question.ipynb', 'w02_ml_task_framing.ipynb', 'w03_feature_leakage_check.ipynb', 'w05_model.ipynb']


In [3]:
for root, dirs, files in os.walk(".."):
    for f in files:
        if f == "content_refresh_anonymized.csv":
            print(os.path.join(root, f))

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1: CTR vs position_tier
Verdict: CONFIRMED

Mean CTR rises monotonically as position improves: deep (n=1,319) 0.15%,
page_3_5 (n=7,242) 0.22%, striking (n=7,304) 0.32%, page_1 (n=11,814) 0.65%,
top_3 (n=2,321) 1.48%. This is the expected relationship — better ranking
positions get clicked more — and every tier here has a reasonable sample
size (all n > 1,300), so this isn't a small-sample artifact. This confirms
position is a real driver behind the CTR-fix logic: pages sitting in
page_1/top_3 with weak CTR are the ones worth flagging, since the position
itself proves the content is relevant enough to rank — a low CTR there
points at a fixable presentation problem (title/meta), not a relevance
problem.

### Signal 2: CTR vs impression_tier
Verdict: MIXED

CTR rises from moderate (n=10,469) 0.21% → good (n=7,205) 0.31% →
excellent (n=1,078) 0.31% — the direction you'd expect if volume signals
demand/relevance. But low (n=11,248) breaks the pattern with the highest
mean CTR of all, 0.94%. That's not what a simple "more impressions = more
opportunity" story predicts. Likely explanation: low-impression pages are
often long-tail, highly specific queries with strong intent match, so a
smaller audience clicks at a higher rate — whereas moderate/good/excellent
tiers include more competitive, broader queries where a page is one of
many options. Because of this, I'm not using impression_tier alone as a
clean linear signal in my rule — it needs to be paired with something
about intent or competition to be trustworthy on its own.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# --- Signal 1: CTR vs position_tier ---
sig1 = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).sort_values("mean_ctr")
print(sig1)

# Verdict: eyeball whether top_3/page_1 rows show meaningfully higher CTR
# than deep/page_3_5. Note the volume floor warning from the dictionary —
# top_3 median volume is tiny (~53 impressions/90d), so one click swings
# CTR ~1.9pp. Say that explicitly in your verdict reasoning.

# --- Signal 2: CTR vs impression_tier ---
sig2 = df.groupby("impression_tier").agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).sort_values("mean_ctr")
print(sig2)

# Verdict: does CTR actually improve as impression volume rises, or do
# "good"/"excellent" tiers still show a CTR gap? That gap IS the opportunity.


                   n  mean_ctr
position_tier                 
deep            1319  0.150212
page_3_5        7242  0.222484
striking        7304  0.323239
page_1         11814  0.652467
top_3           2321  1.483611
                     n  mean_ctr
impression_tier                 
moderate         10469  0.212453
good              7205  0.308314
excellent         1078  0.312662
low              11248  0.937000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
def score_row(row):
    # Simple, readable rule: reward volume, penalize CTR below what its
    # position tier should support. Tune the weights after you see sig1/sig2.
    score = (row["impressions_90d"] * 0.001) - (row["ctr"] * 10)
    if row["position_tier"] in ["top_3", "page_1"] and row["ctr"] < df["ctr"].median():
        reason = "STRONG_POSITION_WEAK_CTR"
        action = "REWRITE_TITLE_META"
    elif row["impression_tier"] in ["good", "excellent"] and row["ctr"] < df["ctr"].median():
        reason = "HIGH_VOLUME_CTR_GAP"
        action = "QUICK_WIN_CTR_FIX"
    else:
        reason = "LOW_PRIORITY"
        action = "MONITOR"
    return pd.Series([score, reason, action])

df[["baseline_score", "reason_code", "action"]] = df.apply(score_row, axis=1)

queue = df.sort_values("baseline_score", ascending=False)
queue[["content_id", "baseline_score", "reason_code", "action"]].to_csv(
    "../outputs/baseline_action_score.csv", index=False
)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

1. **content_5fe46e04994d** — MONITOR — page_1 position but CTR only 0.14% at
   excellent impression volume; scored high purely on volume, not a flagged
   gap. Wrong if this page is inherently low-CTR content (e.g. informational,
   no compelling reason to click through) rather than a fixable issue.

2. **content_aaef01a50def** — MONITOR — page_1, CTR 0.25%, high volume driving
   the score. Wrong if 0.25% is actually normal for this query's intent —
   check main_intent before assuming there's a gap.

3. **content_8c19996aa890** — MONITOR — top_3 position with CTR only 0.15%,
   which is unusually low for a top_3 page (tier average was 1.48%). Wrong if
   this is a MONITOR mislabel — this looks like a candidate for
   STRONG_POSITION_WEAK_CTR that the median cutoff missed.

4. **content_2cb567c3c89b** — MONITOR — page_3_5, CTR 0.10%, low but position
   isn't strong enough to trigger the position-based reason code. Wrong if
   volume alone (excellent tier) should have triggered HIGH_VOLUME_CTR_GAP —
   worth checking why it didn't.

5. **content_4c36c775b818** — MONITOR — top_3, CTR 0.41%, still below the
   tier's 1.48% average despite strong position. Wrong if this CTR is
   actually fine for its specific query type.

6. **content_2dba2b1f9536** — MONITOR — page_3_5, CTR 0.21%, mid-range,
   scored high mostly on volume. Wrong if this page has no real CTR problem
   relative to its position tier.

7. **content_1a9e894be2e2** — MONITOR — page_1, CTR 0.23%, below the page_1
   tier average (0.65%). Wrong if the gap here is intent-driven rather than
   presentation-driven (e.g. branded/navigational query).

8. **content_db5989a78dd3** — MONITOR — page_1, CTR 0.21%, notably below tier
   average. Wrong if the rule's median threshold is set too low to catch this
   as a real gap — worth a manual look.

9. **content_2c2606c5d176** — MONITOR — page_1, CTR 0.53%, close to tier
   average (0.65%), so likely correctly not flagged. Wrong if this ranking
   this high is mostly a volume artifact rather than a real signal of
   opportunity.

10. **content_cb112fce36be** — MONITOR — page_1, CTR 0.16%, well below tier
    average. Wrong if this should have been caught as STRONG_POSITION_WEAK_CTR
    — another case worth checking against the actual median value used.

11. **content_44e481c8f55b** — MONITOR — top_3, CTR 0.65%, below tier average
    (1.48%) but not flagged. Wrong if the median cutoff is too permissive to
    catch top_3 underperformers.

12. **content_9532f197bbc8** — MONITOR — top_3, CTR 0.87%, close to tier
    average, correctly likely not a gap. Wrong if high score here is really
    just volume, not opportunity.

13. **content_36ff89c8214e** — REWRITE_TITLE_META — page_1, CTR only 0.05%,
    a genuinely large gap versus the 0.65% tier average. This is the type of
    row the rule is designed to catch. Wrong if this page's low CTR is due to
    a navigational/branded intent where clicks are inherently rare regardless
    of title — check main_intent.

14. **content_b28d1efd668f** — QUICK_WIN_CTR_FIX — page_3_5, CTR 0.06%, excellent
    impression volume — a real case of traffic sitting behind a poor CTR.
    Wrong if this page's content_type or intent explains the low CTR
    structurally (e.g. it's a reference page not meant to be clicked from
    search).

15. **content_8e7ba84a972b** — MONITOR — page_1, CTR 0.92%, above tier average,
    correctly not flagged. Wrong if this ranks this high purely on volume with
    no actual signal behind it.

16. **content_8451fc6f034d** — REWRITE_TITLE_META — top_3, CTR only 0.03%, an
    extreme gap against the 1.48% tier average — the strongest case in this
    top-20 for a real fix. Wrong if there's a technical reason (e.g. rich
    snippet stealing clicks) rather than a title/meta problem.

17. **content_89e84d699e9e** — MONITOR — page_1, CTR 0.89%, above tier average,
    correctly not flagged. Wrong if it's here mostly because of volume rather
    than a genuine "healthy" signal.

18. **content_aa4baf490b43** — MONITOR — page_1, CTR 0.50%, close to tier
    average, reasonable to leave unflagged. Wrong if volume alone shouldn't
    have ranked it this high in the queue.

19. **content_008fb02c46cb** — MONITOR — page_1, CTR 0.26%, below tier average.
    Wrong if this is another case the median cutoff should have caught as a
    gap but didn't.

20. **content_813e88069237** — QUICK_WIN_CTR_FIX — page_3_5, CTR 0.06%,
    excellent volume, correctly flagged. Wrong if this is a duplicate/near-
    duplicate of row 14's pattern rather than an independent finding worth
    listing twice.

In [6]:
top20 = queue.head(20)[["content_id", "action", "reason_code", "baseline_score", "ctr", "position_tier", "impression_tier"]]
top20


,content_id,action,reason_code,baseline_score,ctr,position_tier,impression_tier
6653,content_5fe46e04994d,MONITOR,LOW_PRIORITY,516.315,0.14,page_1,excellent
17812,content_aaef01a50def,MONITOR,LOW_PRIORITY,514.609,0.25,page_1,excellent
26844,content_8c19996aa890,MONITOR,LOW_PRIORITY,507.752,0.15,top_3,excellent
19636,content_2cb567c3c89b,MONITOR,LOW_PRIORITY,496.727,0.10,page_3_5,excellent
21819,content_4c36c775b818,MONITOR,LOW_PRIORITY,459.003,0.41,top_3,excellent
29400,content_2dba2b1f9536,MONITOR,LOW_PRIORITY,441.334,0.21,page_3_5,excellent
29879,content_1a9e894be2e2,MONITOR,LOW_PRIORITY,413.880,0.23,page_1,excellent
18870,content_db5989a78dd3,MONITOR,LOW_PRIORITY,343.011,0.21,page_1,excellent
13537,content_2c2606c5d176,MONITOR,LOW_PRIORITY,342.099,0.53,page_1,excellent
26531,content_cb112fce36be,MONITOR,LOW_PRIORITY,308.310,0.16,page_1,excellent


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks review

These are the 10 highest-volume rows my rule scored as LOW_PRIORITY —
sorted by impressions_90d to check whether high-traffic pages are being
missed.

Several of these are concerning: content_8c19996aa890 (top_3, ctr=0.15,
509,252 impressions) and content_2cb567c3c89b (page_3_5, ctr=0.10, 497,727
impressions) have CTR well below their position tier's average (top_3
averages 1.48%; page_3_5 averages 0.22%, though 0.10 is still on the low
end) combined with very high volume — yet neither triggered
STRONG_POSITION_WEAK_CTR or HIGH_VOLUME_CTR_GAP. Given both are
impression_tier = excellent, I'd expect HIGH_VOLUME_CTR_GAP to have fired
if ctr < ctr_median, which raises a question: either these rows' CTR isn't
actually below the median I computed (worth printing ctr_median directly
to confirm the real cutoff value), or there's a logic issue in how the
elif chain evaluates these rows.

This exposes a real limitation rather than confirming the rule works:
my rule may be under-flagging some of the highest-opportunity pages in
the entire dataset — the ones with both huge volume and weak CTR are
exactly what a refresh/rewrite program should prioritize first, and
right now they're landing in MONITOR. If I had more time, I'd print
ctr_median explicitly and re-check these specific rows against it by hand
to confirm whether this is a genuine rule miss or my expectation was
wrong.

In [7]:
# Weak picks: bottom of the ranked queue, or rows your rule scored as
# LOW_PRIORITY/MONITOR despite having real traffic
weak = queue[queue["reason_code"] == "LOW_PRIORITY"].sort_values(
    "impressions_90d", ascending=False
).head(10)

weak[["content_id", "action", "reason_code", "baseline_score", "ctr", "position_tier", "impression_tier", "impressions_90d"]]


,content_id,action,reason_code,baseline_score,ctr,position_tier,impression_tier,impressions_90d
6653,content_5fe46e04994d,MONITOR,LOW_PRIORITY,516.315,0.14,page_1,excellent,517715
17812,content_aaef01a50def,MONITOR,LOW_PRIORITY,514.609,0.25,page_1,excellent,517109
26844,content_8c19996aa890,MONITOR,LOW_PRIORITY,507.752,0.15,top_3,excellent,509252
19636,content_2cb567c3c89b,MONITOR,LOW_PRIORITY,496.727,0.10,page_3_5,excellent,497727
21819,content_4c36c775b818,MONITOR,LOW_PRIORITY,459.003,0.41,top_3,excellent,463103
29400,content_2dba2b1f9536,MONITOR,LOW_PRIORITY,441.334,0.21,page_3_5,excellent,443434
29879,content_1a9e894be2e2,MONITOR,LOW_PRIORITY,413.880,0.23,page_1,excellent,416180
13537,content_2c2606c5d176,MONITOR,LOW_PRIORITY,342.099,0.53,page_1,excellent,347399
18870,content_db5989a78dd3,MONITOR,LOW_PRIORITY,343.011,0.21,page_1,excellent,345111
14090,content_44e481c8f55b,MONITOR,LOW_PRIORITY,306.194,0.65,top_3,excellent,312694


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.